# 🛡️ Content Moderation

**Hook:** *"AI content moderation in <100 lines of code"*

Catch sophisticated bad actors asking for money on dating apps. Watch Fenic detect context-dependent violations that keyword filters miss.

In [ ]:
!pip uninstall -y sklearn-compat ibis-framework imbalanced-learn
!pip install fenic matplotlib seaborn polars==1.30.0

In [ ]:
import os 
import getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [ ]:
import fenic as fc
from pydantic import BaseModel, Field

# ⚡ Setup for AI content moderation
session = fc.Session.get_or_create(fc.SessionConfig(
    app_name="content_moderation_demo",
    semantic=fc.SemanticConfig(
        language_models={
            "smart": fc.OpenAILanguageModel(model_name="gpt-4o", rpm=100, tpm=100_000)
        }
    )
))

print("✅ Configured with GPT-4o for sophisticated content analysis")

## 📱 Step 1: Structured Moderation Schema

Define exactly what we want to extract from each message for safety analysis:

In [ ]:
# 🎯 Structured moderation schema using Pydantic
class ModerationResult(BaseModel):
    is_safe: bool = Field(description="Whether content is safe for the platform")
    violation_type: str = Field(description="Type: spam, scam, harassment, or safe")
    confidence: float = Field(description="Confidence 0.0-1.0")
    reason: str = Field(description="Brief explanation")

print("✅ Moderation schema defined:")
print("   • is_safe: Boolean safety flag")
print("   • violation_type: Category of violation")  
print("   • confidence: AI confidence score")
print("   • reason: Human-readable explanation")

## 💬 Step 2: Dating App Messages (Including Scams)

Mix of legitimate messages and sophisticated scams that keyword filters would miss:

In [ ]:
# 📱 Dating app messages with sophisticated scams
messages = session.create_dataframe([
    {"id": "M1", "content": "Hey beautiful! Want to chat privately? I can help you make easy money from home 💰"},
    {"id": "M2", "content": "This restaurant has amazing pasta! Perfect for a romantic dinner 🍝❤️"},
    {"id": "M3", "content": "I'm a successful entrepreneur. Own multiple properties. Let's grab coffee!"},
    {"id": "M4", "content": "You seem special. Here's my crypto investment strategy that made me rich..."},
    {"id": "M5", "content": "You're ugly and stupid. Nobody will ever love you. Why don't you just disappear already?"},
    {"id": "M6", "content": "CLICK HERE NOW! Amazing singles in your area! Meet them tonight! Limited time offer! ACT FAST!!!"}
])

print("📱 Dating App Messages:")
messages.show()
print("\nNotice: Mix of legitimate messages, scams, harassment, and spam!")

## 🛡️ Step 3: AI Content Moderation

Extract structured moderation results using semantic.extract with our schema:

In [ ]:
# 🛡️ AI moderation using structured extraction
moderated = messages.select(
    "id",
    fc.semantic.extract(
        "content",
        ModerationResult,
        model_alias="smart"
    ).alias("moderation")
).cache()  # Cache moderation results

results = moderated.select(
    "id",
    moderated.moderation.is_safe.alias("safe"),
    moderated.moderation.violation_type.alias("violation"),
    moderated.moderation.confidence.alias("confidence"),
    moderated.moderation.reason.alias("reason")
)

print("🛡️ AI MODERATION RESULTS:")
results.show()

unsafe_count = results.filter(fc.col("safe") == False).count()
print(f"\n🎯 CAUGHT {unsafe_count} unsafe messages!")
print("   💰 Money scam detected")
print("   🪙 Crypto pitch flagged") 
print("   😡 Harassment caught")
print("   📢 Spam identified")
print("   🍝 Restaurant recommendation approved")
print("\n💡 AI understands context and intent that keyword filters miss!")

In [ ]:
session.stop()